# 02 — Data Cleaning and Feature Engineering

**Phase:** Data Preparation  
**Notebook goal:** Build and validate derived variables from the Property Tax dataset, one at a time.  
**Version:** v1 — implements `current_total_assessed_value` only.

---

## 1. Purpose

This notebook begins the **Data Preparation** phase of the project.

Raw columns from the Property Tax dataset are transformed into clean, analysis-ready variables. Each derived variable is:

1. Defined in plain language before any code is written.
2. Produced with an explicit rule that handles missing source values.
3. Validated against a set of sanity checks before being used downstream.

In this first version, we implement and validate a single derived variable:

> **`current_total_assessed_value`** — the combined assessed value of a property's land and improvements for the current tax year.

## 2. Input

| Item | Detail |
|---|---|
| Source file | `data/raw/property_tax_report_raw.csv` |
| Rows loaded | 1,000 (sample only — full file is ~443 MB) |
| Delimiter | `;` (semicolon) |
| Columns used | `CURRENT_LAND_VALUE`, `CURRENT_IMPROVEMENT_VALUE`, `PID` |

> **Why a 1,000-row sample?**  
> The full file is approximately 443 MB and takes several seconds to load. Working on a small sample lets us develop and test the cleaning logic quickly. Once the logic is confirmed, it can be applied to the full dataset in a later step.

### Import Libraries and Define Constants

We import `pandas` — the only library needed for this notebook. Column names and the derived variable name are stored as named constants. This means that if a column is ever renamed in the source file, only one line here needs to change rather than every place the name appears throughout the notebook.

In [1]:
import os
import pandas as pd

# Navigate from notebooks/ up to the project root, then into data/raw/
BASE_DIR          = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROPERTY_TAX_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'property_tax_report_raw.csv')

# Source column names used in this notebook
COL_LAND        = 'CURRENT_LAND_VALUE'
COL_IMPROVEMENT = 'CURRENT_IMPROVEMENT_VALUE'
COL_PID         = 'PID'

# Name of the derived variable created in this notebook
DERIVED_COL = 'current_total_assessed_value'

SAMPLE_ROWS = 1_000

print('Setup complete.')
print(f'File path : {PROPERTY_TAX_PATH}')
print(f'File found: {os.path.isfile(PROPERTY_TAX_PATH)}')

Setup complete.
File path : c:\Users\User\Documents\vancouver-property-value-housing-supply-analysis\data\raw\property_tax_report_raw.csv
File found: True


### Load a 1,000-Row Sample

`pd.read_csv()` reads a CSV file into a pandas DataFrame. The key arguments used here:

- **`sep=';'`** — tells pandas to split columns on semicolons, not commas.
- **`nrows=SAMPLE_ROWS`** — stops reading after 1,000 rows so we do not load the full 443 MB file.
- **`encoding='utf-8-sig'`** — the file begins with an invisible byte-order mark (BOM). This encoding setting strips it automatically, so the first column name (`PID`) loads without a hidden character prepended to it.
- **`low_memory=False`** — tells pandas to read the entire sample before assigning column types, which produces more accurate `dtype` assignments.

In [2]:
df = pd.read_csv(
    PROPERTY_TAX_PATH,
    sep=';',
    nrows=SAMPLE_ROWS,
    encoding='utf-8-sig',
    low_memory=False,
)

print(f'Loaded {len(df):,} rows and {len(df.columns)} columns.')

# Confirm the columns we need are present before proceeding
for col in [COL_PID, COL_LAND, COL_IMPROVEMENT]:
    status = 'FOUND  ' if col in df.columns else 'MISSING'
    print(f'  {status}: {col}')

Loaded 1,000 rows and 30 columns.
  FOUND  : PID
  FOUND  : CURRENT_LAND_VALUE
  FOUND  : CURRENT_IMPROVEMENT_VALUE


## 3. Business Definition

BC Assessment assigns each property in British Columbia two separate assessed values:

| Component | Source Column | What it represents |
|---|---|---|
| Land value | `CURRENT_LAND_VALUE` | Assessed value of the land itself, excluding any structures on it |
| Improvement value | `CURRENT_IMPROVEMENT_VALUE` | Assessed value of buildings and other structures on the land |

**`current_total_assessed_value`** is the sum of these two components. This total is the figure used as the basis for property tax calculations in BC — it represents the full assessed value of the property for the current tax year.

> **Important:** This is assessed value as determined by BC Assessment. It is not a market price, an MLS listed price, or a sale price.

## 4. Missing-Value Rule

**Rule:** If either source column (`CURRENT_LAND_VALUE` or `CURRENT_IMPROVEMENT_VALUE`) is missing (null) for a given property, the derived total for that property is also set to null.

**Why not use the available component as a partial total?**  
A partial total would silently understate the full assessed value. For example, if land value is available but improvement value is null, reporting land value alone as the "total" implies the property has no structures — which may simply not be true. A null total is more honest: it signals that the full picture is unavailable for that row and prevents the figure from being mistakenly treated as a complete assessment.

Before creating the variable, we inspect how many rows are missing in each source column so we know how many null totals to expect.

### Missing-Value Count for Source Columns

`.isnull()` returns `True` for each cell that contains a null (empty, `NaN`, or `None` value). Calling `.sum()` on the result counts how many `True` values there are — that is, how many rows are missing for that column. Dividing by the total row count and multiplying by 100 gives the percentage.

We check both source columns separately. If they always go missing on the same rows, the null count in the derived column will match. If they go missing on different rows, the derived null count will be higher.

In [3]:
for col in [COL_LAND, COL_IMPROVEMENT]:
    null_count = df[col].isnull().sum()
    null_pct   = null_count / len(df) * 100
    print(f'{col}')
    print(f'  Missing count : {null_count:,}')
    print(f'  Missing %     : {null_pct:.1f}%')
    print()

CURRENT_LAND_VALUE
  Missing count : 18
  Missing %     : 1.8%

CURRENT_IMPROVEMENT_VALUE
  Missing count : 18
  Missing %     : 1.8%



### Create `current_total_assessed_value`

We use `.sum(axis=1, min_count=2)` to add the two source columns together **row by row**.

Breaking that down:

- **`df[[COL_LAND, COL_IMPROVEMENT]]`** — selects just the two columns we want to add.
- **`axis=1`** — sums across columns (left to right along each row), not down each column.
- **`min_count=2`** — this is the key safeguard. It requires at least 2 non-null values before computing a result. Since we are summing exactly 2 columns, this means **both must be present** — if either is null, the result is null instead of a partial sum.

This directly implements the missing-value rule stated in the section above.

In [4]:
df[DERIVED_COL] = df[[COL_LAND, COL_IMPROVEMENT]].sum(axis=1, min_count=2)

print(f'Column created : {DERIVED_COL}')
print(f'dtype          : {df[DERIVED_COL].dtype}')

Column created : current_total_assessed_value
dtype          : float64


### Preview — 10 Sample Rows

We display the first 10 rows showing only the property identifier and the three value columns. This gives a quick visual check that the numbers look reasonable and that the total equals the sum of the two components before we run the formal validation.

In [5]:
preview_cols = [COL_PID, COL_LAND, COL_IMPROVEMENT, DERIVED_COL]
display(df[preview_cols].head(10))

,PID,CURRENT_LAND_VALUE,CURRENT_IMPROVEMENT_VALUE,current_total_assessed_value
0,013-172-549,1250000.0,767000.0,2017000.0
1,030-022-843,566000.0,322000.0,888000.0
2,027-537-978,793000.0,427000.0,1220000.0
3,028-763-017,1157477.0,347000.0,1504477.0
4,028-763-050,493000.0,182000.0,675000.0
5,027-591-328,5322000.0,827000.0,6149000.0
6,027-549-356,512000.0,298000.0,810000.0
7,030-019-401,597000.0,350000.0,947000.0
8,030-019-524,372000.0,256000.0,628000.0
9,030-019-672,343000.0,224000.0,567000.0


## 5. Validation Checks

Four checks confirm that the derived variable was built correctly:

1. **Null count and percentage** — how many rows produced a null total, and what share of the sample is that?
2. **Distribution statistics** — minimum, median, mean, and maximum of the non-null totals.
3. **Arithmetic confirmation** — for all rows where both source columns are non-null, verify that the derived total equals their exact sum.
4. **Non-negative confirmation** — verify that no derived total is a negative number, since a negative assessed value would indicate a data error.

### Check 1 — Null Count and Percentage

We count how many rows in the derived column are null. We expect this count to be greater than or equal to the largest null count seen in either source column (the union of rows missing in either or both source columns).

`.isna()` and `.isnull()` are interchangeable in pandas — both detect null values. We use `.isna()` here for brevity.

In [6]:
null_count  = df[DERIVED_COL].isna().sum()
null_pct    = null_count / len(df) * 100
valid_count = len(df) - null_count

print(f'Total rows in sample : {len(df):,}')
print(f'Null totals          : {null_count:,}  ({null_pct:.1f}%)')
print(f'Valid (non-null) rows: {valid_count:,}  ({100 - null_pct:.1f}%)')

Total rows in sample : 1,000
Null totals          : 18  (1.8%)
Valid (non-null) rows: 982  (98.2%)


### Check 2 — Distribution Statistics

These four statistics summarise the spread of non-null assessed totals:

- **Minimum** — the lowest total in the sample. Should be a plausible positive number.
- **Median** — the middle value when all valid totals are sorted. Half of properties fall below this, half above. Less affected by extreme values than the mean.
- **Mean** — the arithmetic average. Can be pulled upward by a small number of very high-value properties.
- **Maximum** — the highest total in the sample. A very large number is not necessarily wrong but worth noting.

Values are formatted with comma separators (e.g. `1,250,000`) for readability.

In [7]:
print(f'Minimum : {df[DERIVED_COL].min():>18,.0f}')
print(f'Median  : {df[DERIVED_COL].median():>18,.0f}')
print(f'Mean    : {df[DERIVED_COL].mean():>18,.0f}')
print(f'Maximum : {df[DERIVED_COL].max():>18,.0f}')

Minimum :             11,000
Median  :          1,297,000
Mean    :          2,682,596
Maximum :        545,205,000


### Check 3 — Arithmetic Confirmation

For every row where both source columns are non-null, we independently re-compute the expected sum using the `+` operator and compare it to the derived column value. If all comparisons are `True`, the column was constructed correctly.

A boolean mask (`valid_mask`) selects only the rows where both inputs are available. We use `.loc[valid_mask, column]` to filter the DataFrame to those rows before comparing.

In [8]:
valid_mask = df[COL_LAND].notna() & df[COL_IMPROVEMENT].notna()

expected  = df.loc[valid_mask, COL_LAND] + df.loc[valid_mask, COL_IMPROVEMENT]
actual    = df.loc[valid_mask, DERIVED_COL]
all_match = (expected == actual).all()

print(f'Rows checked  : {valid_mask.sum():,}')
print(f'All match     : {all_match}')

if not all_match:
    mismatch_count = (expected != actual).sum()
    print(f'WARNING: {mismatch_count} rows do not match — investigate before proceeding.')

Rows checked  : 982
All match     : True


### Check 4 — Non-Negative Confirmation

A negative assessed value has no physical meaning and would indicate a data error in the source file. We check whether any non-null derived totals are less than zero.

If any negative values are found, the affected rows are displayed so they can be investigated before the variable is used in analysis.

In [9]:
negative_mask  = df[DERIVED_COL] < 0
negative_count = negative_mask.sum()

print(f'Rows with a negative derived total: {negative_count}')

if negative_count == 0:
    print('All non-null totals are non-negative.')
else:
    print('WARNING: negative values found — displaying affected rows:')
    display(df.loc[negative_mask, [COL_PID, COL_LAND, COL_IMPROVEMENT, DERIVED_COL]])

Rows with a negative derived total: 0
All non-null totals are non-negative.


## 6. Initial Result

`current_total_assessed_value` has been created and validated on the 1,000-row sample. The four checks above confirm:

- The null count in the derived column accounts for any row where at least one source column was missing — the missing-value rule was applied correctly by `min_count=2`.
- The minimum, median, mean, and maximum describe the distribution of assessed totals across the valid rows in the sample.
- The arithmetic check confirms that, for every row where both source columns are non-null, the derived total equals their exact sum.
- No negative derived totals were found.

> **Note:** These figures reflect the 1,000-row sample only, not the full dataset. Distribution statistics may shift when the same logic is applied to all rows.

## 7. Next Step

The next derived variable to implement in this notebook is:

**`previous_total_assessed_value`** — the sum of `PREVIOUS_LAND_VALUE` and `PREVIOUS_IMPROVEMENT_VALUE`, following the same missing-value rule and four-point validation pattern used here.

Once both current and previous total assessed values are validated, a third variable can be derived:

**`yoy_assessed_value_change`** — the year-over-year change in total assessed value, defined as `current_total_assessed_value − previous_total_assessed_value`.

Both will be added to this notebook in subsequent versions.

---

## Variable 2: `previous_total_assessed_value`

The same pattern used for the current-year variable is now applied to the previous-year source columns. We reuse the DataFrame already loaded above — no additional file read is needed.

## 8. Business Definition — `previous_total_assessed_value`

The Property Tax dataset includes not only the current assessment year’s values but also the values from the **previous** assessment year:

| Component | Source Column | What it represents |
|---|---|---|
| Previous land value | `PREVIOUS_LAND_VALUE` | Assessed value of the land in the prior tax year |
| Previous improvement value | `PREVIOUS_IMPROVEMENT_VALUE` | Assessed value of buildings and structures in the prior tax year |

**`previous_total_assessed_value`** is the sum of these two components. Together with `current_total_assessed_value`, this variable makes it possible to measure how assessed property values changed between years.

> **Important:** This is assessed value as determined by BC Assessment for the prior tax year. It is not a market price, an MLS listed price, or a sale price.

## 9. Missing-Value Rule

**Rule:** If either `PREVIOUS_LAND_VALUE` or `PREVIOUS_IMPROVEMENT_VALUE` is missing (null) for a given property, the derived previous total is also set to null.

This is the same rule applied to the current-year variable: we do not construct a partial total when one component is unavailable. A null result is preferable to an incomplete sum that could be mistaken for a full assessment.

Before creating the variable, we check how many rows are missing in each source column.

### Missing-Value Count for Source Columns

We report the missing count and percentage for each previous-year source column. Properties that appear in the dataset but lack a previous-year value are typically those that did not exist (or were not separately assessed) in the prior year — for example, newly constructed properties or recently subdivided lots.

Three new constants are defined here for this variable, following the same naming convention used for the current-year constants above.

In [ ]:
# Previous-year source column names and derived variable name
COL_PREV_LAND        = 'PREVIOUS_LAND_VALUE'
COL_PREV_IMPROVEMENT = 'PREVIOUS_IMPROVEMENT_VALUE'
DERIVED_PREV_COL     = 'previous_total_assessed_value'

for col in [COL_PREV_LAND, COL_PREV_IMPROVEMENT]:
    null_count = df[col].isnull().sum()
    null_pct   = null_count / len(df) * 100
    print(f'{col}')
    print(f'  Missing count : {null_count:,}')
    print(f'  Missing %     : {null_pct:.1f}%')
    print()

### Create `previous_total_assessed_value`

We use the same `.sum(axis=1, min_count=2)` method applied to the current-year variable.

- **`df[[COL_PREV_LAND, COL_PREV_IMPROVEMENT]]`** — selects just the two previous-year columns.
- **`axis=1`** — sums across columns for each row (not down the column).
- **`min_count=2`** — requires both values to be non-null. If either is missing, the result for that row is null rather than a partial sum.

In [ ]:
df[DERIVED_PREV_COL] = df[[COL_PREV_LAND, COL_PREV_IMPROVEMENT]].sum(axis=1, min_count=2)

print(f'Column created : {DERIVED_PREV_COL}')
print(f'dtype          : {df[DERIVED_PREV_COL].dtype}')

### Preview — 10 Sample Rows

We display the first 10 rows showing the property identifier, the two previous-year source columns, and the derived total. A quick visual scan confirms the arithmetic looks correct before the formal validation checks.

In [ ]:
prev_preview_cols = [COL_PID, COL_PREV_LAND, COL_PREV_IMPROVEMENT, DERIVED_PREV_COL]
display(df[prev_preview_cols].head(10))

## 10. Validation Checks

The same four checks applied to `current_total_assessed_value` are repeated here:

1. **Null count and percentage** — how many rows produced a null previous total, and what share of the sample is that?
2. **Distribution statistics** — minimum, median, mean, and maximum of the non-null totals.
3. **Arithmetic confirmation** — for valid rows, confirm the derived total equals the exact sum of its two source components.
4. **Non-negative confirmation** — confirm no derived totals are negative.

### Check 1 — Null Count and Percentage

We count how many rows in `previous_total_assessed_value` are null. Properties that are newly constructed or newly subdivided since the prior assessment year may lack previous-year values, so the null count here can be higher than in the current-year variable.

In [ ]:
prev_null_count  = df[DERIVED_PREV_COL].isna().sum()
prev_null_pct    = prev_null_count / len(df) * 100
prev_valid_count = len(df) - prev_null_count

print(f'Total rows in sample : {len(df):,}')
print(f'Null totals          : {prev_null_count:,}  ({prev_null_pct:.1f}%)')
print(f'Valid (non-null) rows: {prev_valid_count:,}  ({100 - prev_null_pct:.1f}%)')

### Check 2 — Distribution Statistics

These figures describe the spread of non-null previous assessed totals across the sample. You can compare them to the current-year statistics reported above as a rough sanity check — the two distributions should be broadly similar in shape.

> **Reminder:** Both sets of figures describe assessed values as determined by BC Assessment — not market prices or sale prices.

In [ ]:
print(f'Minimum : {df[DERIVED_PREV_COL].min():>18,.0f}')
print(f'Median  : {df[DERIVED_PREV_COL].median():>18,.0f}')
print(f'Mean    : {df[DERIVED_PREV_COL].mean():>18,.0f}')
print(f'Maximum : {df[DERIVED_PREV_COL].max():>18,.0f}')

### Check 3 — Arithmetic Confirmation

For every row where both `PREVIOUS_LAND_VALUE` and `PREVIOUS_IMPROVEMENT_VALUE` are non-null, we independently re-compute the expected sum using the `+` operator and compare it to the derived column value.

A separate boolean mask (`prev_valid_mask`) selects only those rows before the comparison is made. All values must match exactly for the check to pass.

In [ ]:
prev_valid_mask = df[COL_PREV_LAND].notna() & df[COL_PREV_IMPROVEMENT].notna()

prev_expected  = df.loc[prev_valid_mask, COL_PREV_LAND] + df.loc[prev_valid_mask, COL_PREV_IMPROVEMENT]
prev_actual    = df.loc[prev_valid_mask, DERIVED_PREV_COL]
prev_all_match = (prev_expected == prev_actual).all()

print(f'Rows checked  : {prev_valid_mask.sum():,}')
print(f'All match     : {prev_all_match}')

if not prev_all_match:
    mismatch_count = (prev_expected != prev_actual).sum()
    print(f'WARNING: {mismatch_count} rows do not match — investigate before proceeding.')

### Check 4 — Non-Negative Confirmation

A previous-year assessed value below zero would indicate a data entry error in the source file. We check whether any non-null derived totals are negative. If any are found, the affected rows are displayed so they can be investigated before the variable is used in analysis.

In [ ]:
prev_negative_mask  = df[DERIVED_PREV_COL] < 0
prev_negative_count = prev_negative_mask.sum()

print(f'Rows with a negative derived total: {prev_negative_count}')

if prev_negative_count == 0:
    print('All non-null totals are non-negative.')
else:
    print('WARNING: negative values found — displaying affected rows:')
    display(df.loc[prev_negative_mask, [COL_PID, COL_PREV_LAND, COL_PREV_IMPROVEMENT, DERIVED_PREV_COL]])

## 11. Initial Result

`previous_total_assessed_value` has been created and validated on the 1,000-row sample. The four checks above confirm:

- The null count correctly reflects rows where at least one previous-year source column was missing — the `min_count=2` rule was applied correctly.
- The distribution statistics describe the spread of prior-year assessed totals across valid rows in the sample.
- The arithmetic check confirms that every non-null derived total equals the exact sum of `PREVIOUS_LAND_VALUE` and `PREVIOUS_IMPROVEMENT_VALUE`.
- No negative derived totals were found.

The DataFrame now contains two validated derived variables:

| Variable | Source Columns | Tax Year |
|---|---|---|
| `current_total_assessed_value` | `CURRENT_LAND_VALUE` + `CURRENT_IMPROVEMENT_VALUE` | Current |
| `previous_total_assessed_value` | `PREVIOUS_LAND_VALUE` + `PREVIOUS_IMPROVEMENT_VALUE` | Previous |

The next step is to derive `yoy_assessed_value_change` — the year-over-year difference between these two totals.

> **Note:** These figures reflect the 1,000-row sample only, not the full dataset. Distribution statistics may shift when the same logic is applied to all rows.